In [1]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp  # type: ignore[import]
from pathlib import Path
import matplotlib.pyplot as plt
import cProfile
import pstats
# --------------------------------- Internal imports --------------------------------- #
from radiopyo.chemistry.base_chemistry import ReactionRateEnum
from radiopyo.physics.beam import ConstantBeam, ParticleBeam, PulsedBeam
from radiopyo.sim_env import SimEnv
from radiopyo.ode_solver import derive, jac, derive2
import radiopyo as rp


In [6]:
file = Path(rf"{rp.__path__[0]}\data\reactions.ron")
beam = PulsedBeam("e",1000, 500e-6, 50e-6, max_dose=2)
#beam = ConstantBeam("e",1e3, max_dose=2)
env = SimEnv.from_ron_file(file)
t_span = [1e-7, 1]
yinit = env.initial_values()
method = "LSODA"


In [11]:
def solve_ode():
    sol = solve_ivp(derive,
                    t_span,
                    yinit,
                    method=method,
                    rtol=1e-6,
                    args=(env, beam),
                    jac=jac,
                    max_step=100e-6,
                    )
profiler = cProfile.Profile()
cProfile.run("solve_ode()", "time_profile.prof")
p = pstats.Stats("time_profile.prof")
p = p.sort_stats("cumtime").print_stats("radiopyo")

Mon Mar 13 18:55:57 2023    time_profile.prof

         4656009 function calls (4656007 primitive calls) in 6.427 seconds

   Ordered by: cumulative time
   List reduced from 109 to 36 due to restriction <'radiopyo'>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    17636    1.174    0.000    5.335    0.000 C:\Users\rtonneau\DEV\PYTHON\radiopyo\radiopyo\ode_solver.py:55(derive)
    17636    0.017    0.000    2.888    0.000 C:\Users\rtonneau\DEV\PYTHON\radiopyo\radiopyo\sim_env.py:169(compute_reactions)
    17636    0.231    0.000    2.870    0.000 C:\Users\rtonneau\DEV\PYTHON\radiopyo\radiopyo\sim_env.py:182(<listcomp>)
   440900    2.392    0.000    2.569    0.000 C:\Users\rtonneau\DEV\PYTHON\radiopyo\radiopyo\chemistry\k_reactions.py:106(compute_reaction)
    18720    0.250    0.000    1.132    0.000 C:\Users\rtonneau\DEV\PYTHON\radiopyo\radiopyo\sim_env.py:144(update_cc_species)
    18720    0.184    0.000    0.657    0.000 C:\Users\rtonneau\DEV\PYTHON\rad

In [13]:
def solve_ode2():
    sol = solve_ivp(derive2,
                    t_span,
                    yinit,
                    method=method,
                    rtol=1e-6,
                    args=(env, beam),
                    jac=jac,
                    max_step=100e-6,
                    )
profiler = cProfile.Profile()
cProfile.run("solve_ode2()", "time_profile2.prof")
p = pstats.Stats("time_profile2.prof")
p = p.sort_stats("cumtime").print_stats("radiopyo")

Mon Mar 13 18:56:13 2023    time_profile2.prof

         5076984 function calls (5076982 primitive calls) in 5.207 seconds

   Ordered by: cumulative time
   List reduced from 111 to 37 due to restriction <'radiopyo'>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    16224    0.060    0.000    4.358    0.000 C:\Users\rtonneau\DEV\PYTHON\radiopyo\radiopyo\ode_solver.py:86(derive2)
    16224    0.062    0.000    3.404    0.000 C:\Users\rtonneau\DEV\PYTHON\radiopyo\radiopyo\sim_env.py:162(compute_reactions2)
    16224    0.265    0.000    3.089    0.000 C:\Users\rtonneau\DEV\PYTHON\radiopyo\radiopyo\sim_env.py:166(<listcomp>)
   405600    1.609    0.000    2.574    0.000 C:\Users\rtonneau\DEV\PYTHON\radiopyo\radiopyo\chemistry\k_reactions.py:95(compute_reaction2)
    17082    0.175    0.000    0.883    0.000 C:\Users\rtonneau\DEV\PYTHON\radiopyo\radiopyo\sim_env.py:144(update_cc_species)
   405600    0.500    0.000    0.659    0.000 C:\Users\rtonneau\DEV\PYTHON\

In [10]:
!snakeviz time_profile2.prof

^C
